## 可视化决策树的方式

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn import tree
import graphviz  # Graphviz 是一个开源的图可视化软件，用于绘制决策树结构图

# --- 数据准备阶段 ---
# 加载 iris 数据集：150个样本，4个特征（花萼/花瓣的长度和宽度），3类鸢尾花
dataset = load_iris() 
# 转换成 pandas DataFrame 形式，便于数据处理
df = pd.DataFrame(dataset.data, columns = dataset.feature_names)
df['Species'] = dataset.target  # 添加品种标签列

# 将数字标签（0/1/2）替换为实际品种名称，便于结果可读
target = np.unique(dataset.target)
target_names = np.unique(dataset.target_names)
targets = dict(zip(target, target_names))
df['Species'] = df['Species'].replace(targets)

# 提取特征数据 X 和标签 y
X = df.drop(columns="Species")
y = df["Species"]
feature_names = X.columns  # 记录特征名称，用于后续可视化
labels = y.unique()        # 记录类别名称

# 划分训练集（60%）和测试集（40%）
X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                 test_size = 0.4,
                                                 random_state = 42)
# 构建决策树分类器：max_depth=3 限制树深度防止过拟合
model = DecisionTreeClassifier(max_depth =3, random_state = 42)
model.fit(X_train, y_train)  # 训练模型

### 文字表示

In [ ]:
# 方法一：以文字形式输出决策树的决策规则
# 树结构清晰展示了从根节点到叶节点的分类路径
# 例如：feature_2 <= 2.45 表示花瓣长度 <= 2.45cm 时直接归为 setosa 类
text_representation = tree.export_text(model)
print(text_representation)

### plot_tree函数

In [ ]:
# 方法二：使用 matplotlib 绘制决策树图形
# feature_names 和 class_names 使节点显示可读的特征名和类别名
# rounded=True：圆角矩形节点；filled=True：用颜色填充反映类别纯度
plt.figure(figsize=(30,10), facecolor ='g')
a = tree.plot_tree(model,
                   feature_names = feature_names,
                   class_names = labels,
                   rounded = True,
                   filled = True,
                   fontsize=14)
plt.show()

### graphviz模块

该模块需要安装graphviz，推荐在ubuntu上```sudo apt install graphviz```或者Mac上```brew install graphviz```

In [ ]:
# 方法三：使用 Graphviz 绘制更精美的决策树
# export_graphviz 将决策树导出为 DOT 格式（一种图形描述语言）
# filled=True 表示用颜色填充节点，颜色深浅反映类别纯度
dot_data = tree.export_graphviz(model, out_file=None, 
                                feature_names=dataset.feature_names,  
                                class_names=dataset.target_names,
                                filled=True)

# 使用 graphviz.Source 解析 DOT 格式并渲染为图片
graph = graphviz.Source(dot_data, format="png") 
graph

### dtreeviz模块

In [28]:
dtreeviz??

In [ ]:
from dtreeviz.trees import *  # dtreeviz：更高级的决策树可视化库，展示特征分布

# 重新准备数据（使用原始数值数组，dtreeviz 要求数值输入）
X = dataset.data
y = dataset.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 重新训练决策树模型
clf = tree.DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)

# 方法四：使用 dtreeviz 进行高级可视化
# dtreeviz 在每个节点处展示该节点的特征分布直方图
# 能直观看到划分点如何将数据分为不同类别
viz = dtreeviz(clf, 
               x_data=X_train,
               y_data=y_train,
               target_name='class',
               feature_names=dataset.feature_names, 
               class_names=list(dataset.target_names), 
               title="Decision Tree - Iris data set")
viz

## sns.heatmap可视化报告

In [ ]:
# 使用热力图(heatmap)可视化分类评估报告
from sklearn.metrics import classification_report
import numpy as np
import seaborn as sns
import pandas as pd

# 在测试集上进行预测
y_pred = model.predict(X_test)
target_names = np.unique(dataset.target_names)

# 打印文本格式的分类报告（包含精确率、召回率、F1分数）
clf_report = classification_report(y_test,
                                   y_pred,
                                   labels=labels,
                                   target_names=target_names)
print(clf_report)

# 将分类报告转为字典格式，再转为 DataFrame 用于热力图绘制
# output_dict=True 使 classification_report 返回字典而非字符串
clf_report = classification_report(y_test,
                                   y_pred,
                                   labels=labels,
                                   target_names=target_names,
                                   output_dict=True)
# iloc[:-1, :] 排除最后一行（support），.T 转置使类别为横轴、指标为纵轴
# annot=True 在每个格子中显示数值
plt.figure(figsize=(8,6))
sns.heatmap(pd.DataFrame(clf_report).iloc[:-1, :].T, annot=True)
plt.show()